环境：Apache Polaris (REST Catalog)-1.3.0 + Spark SQL-3.5.8 + Kyuubi-1.10.3 + MinIO

## 概述
### 为什么需要 Schema 演进
在传统的数仓或早期数据湖格式（如纯 Hive/Parquet）中，表结构（Schema）一旦定义，往往就被视为不可变或极难修改的契约。如果没有灵活的 Schema 演进功能，业务将面临以下巨大痛点：

+ **痛点一：业务变更即停机**
    - 当业务需求变化（例如需要新增一个字段记录用户来源，或者修改某个字段的类型以容纳更大数值）时，传统方式通常要求重写整张表。对于 TB 级甚至 PB 级的数据，这意味着漫长的停机时间、巨大的计算资源消耗，以及在此期间数据服务完全不可用。
+ **痛点二：历史数据与新代码不兼容**
    - 如果强行修改了表结构，旧的历史数据文件可能因为缺少新字段或类型不匹配而无法被新代码读取，导致查询报错。反之，新写入的数据也可能无法被依赖旧结构的报表识别。这种新旧数据割裂使得数据回溯和分析变得极其困难。
+ **痛点三：协作僵化与试错成本高**
    - 由于修改结构的成本太高，开发团队往往不敢轻易调整模型，导致表结构逐渐腐化，充斥着大量无用字段或通过复杂的 JSON 字符串来规避结构限制，最终让数据变得难以理解和维护。

**Iceberg 的解决方案：**

Iceberg 支持表结构的原地演进。您可以像使用 SQL 一样对表结构进行演变——即便是在嵌套结构中也是如此——或者在数据量发生变化时更改分区布局。Iceberg 不需要诸如重写表数据或迁移至新表之类的昂贵操作。

例如，Hive 表的分区设置无法更改，因此从每日分区布局转换到每小时分区布局就需要创建一个新的表。而且由于查询依赖于分区，所以对于新的表，查询必须进行重写。在某些情况下，即使是像重命名列这样简单的更改也可能既不被支持，又可能会导致数据正确性问题。

## 核心机制
+ **列 ID:** 这是 Iceberg Schema 演进的基石。
    - 在传统格式（如 Hive/Parquet）中，列通常按位置（Position）识别。如果我在第 2 列插入一个新列，后面的列位置都会偏移，导致读取旧文件时数据错乱。
    - 在 Iceberg 中，每一列在创建时都会被分配一个全局唯一的整数 ID。无论列在 Schema 中的顺序如何变化，或者列名如何修改，Iceberg 始终通过 `column id` 来映射物理文件中的数据。
+ **快照隔离: **Schema 的变更是原子性的。变更操作会生成一个新的快照。
    - 查询旧快照时，使用旧的 Schema 读取旧文件。
    - 查询新快照时，使用新的 Schema 读取新旧文件（Iceberg 会在内存中自动处理缺失列的填充或类型转换）。
+ **向后兼容:** 新增列时，旧数据文件中该列的值被视为 `NULL`，无需重写旧文件。

---

## 模式演变
Iceberg 支持以下的模式演变：

+ **Add** -- 向表格或嵌套结构中添加一个新列;
+ **Drop** -- 从表中或嵌套结构中删除一个已存在的列;
+ **Rename **-- 重命名嵌套结构体中的现有列或字段
+ **Update **-- 增大列、结构体字段、映射键、映射值或列表元素的类型
+ **Reorder **-- 更改嵌套结构体中列或字段的顺序

Iceberg模式的更新属于元数据变更，因此无需重写数据文件即可完成更新操作。

## 正确性
Iceberg确保了模式演变所带来的变更既相互独立又不会产生副作用，并且无需重新编写文件：

1. 新增的列永远不会从其他列中读取现有值。
2. 删除一个列或字段并不会影响其他任何列中的数据。
3. 更新某一列或字段并不会改变其他任何列中的值。
4. 在结构体中改变列或字段的顺序并不会改变与某一列或字段名称相对应的值。

Iceberg系统使用独特的标识符来追踪表格中的每一列。当您添加新列时，会为其分配一个新的标识符，这样就不会出现误用现有数据的情况。

+ 按名称来追踪列的格式可能会在某个名称被重新使用时无意中删除该列，这违反了第 1 条规定。
+ 按位置排列列的格式若要删除列，则必须更改每个列所使用的名称，而这样做会违反第 2 条规则。

---

## 分区演变
Iceberg表分区可以在现有表中进行更新，因为查询并不直接引用分区值。

在创建新的分区规格时，先前使用旧规格写入的数据将保持不变。新数据则会按照新的规格以新的布局进行写入。每个分区版本的元数据会单独保存。由于这种特性，在开始编写查询时，会遇到分区规划的问题。在这种情况下，每个分区布局会根据其特定的分区布局所衍生的过滤条件来分别规划文件。以下是一个虚构示例的可视化展示：

<!-- 这是一张图片，ocr 内容为： -->
![](https://cdn.nlark.com/yuque/0/2026/png/45387277/1773478779646-0c73baf7-3084-49f6-bf1e-cca8e364c547.png)

2008 年的数据是按月份进行划分的。从 2009 年开始，该表格会进行更新，使得数据按日进行划分。这两种划分方式能够在同一张表格中并存。

Iceberg采用了隐藏分区技术，因此您无需针对特定的分区布局编写查询语句就能实现快速查询。相反，您可以编写选择所需数据的查询语句，而Iceberg会自动裁剪那些不包含匹配数据的文件。

分区演进是一种元数据操作，并且不会立即重写文件。

## 排序顺序演变
与分区规格类似，Iceberg列排序顺序也可以在现有表中进行更新。在更新排序顺序时，以较早顺序写入的旧数据将保持不变。引擎总是可以选择按照最新的排序顺序写入数据，或者在排序成本过高时不进行排序而直接写入未排序的数据。

---

## 实战案例：电商订单表的迭代演进
场景模拟：

1. 初始状态: 订单表只有基础信息。
2. 需求变更 1: 需要记录用户的邮箱和注册时间（添加列）。
3. 需求变更 2: 业务术语变更，`cust_id` 改为 `user_id`（重命名）。
4. 需求变更 3: 金额精度不足，`amount` 从 `FLOAT` 升级为 `DOUBLE`（类型提升）。
5. 需求变更 4: 清理不再使用的 `legacy_flag` 列（删除列）。
6. 验证: 查询历史数据和最新数据，验证演进的正确性。

### 初始化：创建初始表
```sql
-- 1. 创建初始订单表 (v1)
-- 注意：这里故意将 amount 设为 FLOAT，后续演示升级
CREATE TABLE IF NOT EXISTS dataspire_catalog.db_dev.orders_v1 (
    order_id BIGINT COMMENT '订单ID',
    cust_id BIGINT COMMENT '客户ID (旧称)',
    amount FLOAT COMMENT '订单金额',
    status STRING COMMENT '状态',
    dt STRING COMMENT '分区日期'
) USING iceberg
PARTITIONED BY (dt);

-- 2. 插入初始历史数据 (模拟旧数据)
INSERT
INTO
    dataspire_catalog.db_dev.orders_v1
VALUES
    (1001, 501, 99.5, 'PAID', '2026-03-10')
  , (1002, 502, 200.0, 'PENDING', '2026-03-10')
;


-- 3. 查看当前 Schema
DESCRIBE FORMATTED dataspire_catalog.db_dev.orders_v1;
```

---

### 演进操作一：添加列
业务方要求增加 `email` (字符串) 和 `register_time` (时间戳)。我们希望在 `cust_id` 之后插入这些列。

```sql
-- 1. 添加新列
-- Iceberg 允许指定 AFTER 某列，也可以不加 (默认追加到末尾)
ALTER TABLE dataspire_catalog.db_dev.orders_v1
ADD COLUMNS (
    email STRING COMMENT '用户邮箱' AFTER cust_id,
    register_time TIMESTAMP COMMENT '注册时间' AFTER email
);

-- 2. 插入新数据 (包含新列)
INSERT INTO
    dataspire_catalog.db_dev.orders_v1
VALUES
    ( 1003, 503, 150.75, 'PAID', 'alice@example.com'
    , TIMESTAMP('2026-01-01 10:00:00'), '2026-03-11');

-- 3. 验证：查询所有数据
-- 预期：旧数据 (1001, 1002) 的 email 和 register_time 显示为 NULL
--      新数据 (1003) 显示具体值
SELECT
    t.order_id      AS order_id
  , t.cust_id       AS cust_id
  , t.email         AS email
  , t.register_time AS register_time
  , t.amount        AS amount
FROM
    dataspire_catalog.db_dev.orders_v1 t
WHERE
    1 = 1
ORDER BY
    order_id
;
```

原理说明: 此时 MinIO 中的旧 Parquet 文件并没有被修改，它们依然只包含 `order_id`, `cust_id`, `amount`, `status`, `dt`。Iceberg 在读取旧文件时，发现 Schema 中有 `email` 但文件中没有，自动填充 `NULL`。

---

### 演进操作二：重命名与调整顺序
业务规范统一，`cust_id` 更名为 `user_id`。同时我们希望把 `status` 放到 `amount` 前面。

```sql
-- 1. 重命名列 (Column ID 保持不变，仅元数据名称变更)
ALTER TABLE dataspire_catalog.db_dev.orders_v1 RENAME COLUMN cust_id TO user_id;

-- 2. 调整列顺序 (仅影响 DESCRIBE 和 SELECT * 的展示顺序，不影响存储)
-- 注意：Spark SQL 原生 ALTER TABLE 对 reorder 支持有限，通常依赖 ADD COLUMNS 时的位置或特定语法。
-- 在 Iceberg 中，更推荐的做法是通过重建视图或依赖列 ID 的透明性。
-- 但 Iceberg 支持通过 alter table ... rename column 间接调整，或者直接接受顺序变化。
-- 此处演示重命名的核心能力，顺序在 SELECT 中显式指定更佳。

-- 3. 验证：检查 Schema 变化
DESCRIBE dataspire_catalog.db_dev.orders_v1;
-- 此时应看到 user_id 而不是 cust_id
```

---

### 演进操作三：类型提升
发现 `FLOAT` 精度不够，导致金额计算有误差，需要升级为 `DOUBLE`。

```sql
-- 1. 提升列类型 (Safe Promotion: Float -> Double)
ALTER TABLE dataspire_catalog.db_dev.orders_v1 ALTER COLUMN amount TYPE DOUBLE;

-- 2. 插入更高精度的数据
INSERT INTO
    dataspire_catalog.db_dev.orders_v1
(order_id, user_id, amount, status, email, register_time, dt)
VALUES
    (1004, 504, 123.456789012345, 'PAID', 'bob@example.com', TIMESTAMP('2026-02-01 12:00:00'), '2026-03-12')
;

-- 3. 验证：查询金额列
-- 预期：旧数据 (99.5) 和新数据 (123.456...) 都能正常显示，且类型为 Double
SELECT
    order_id       AS order_id
  , amount         AS amount
  , typeof(amount) AS data_type
FROM
    dataspire_catalog.db_dev.orders_v1
ORDER BY
    order_id;
```

原理说明: 旧文件中的 `amount` 依然是 `FLOAT` 二进制格式。Iceberg 读取时，检测到 Schema 要求 `DOUBLE`，会在内存中将 `FLOAT` 值无损转换为 `DOUBLE`。

---

### 演进操作四：删除列
发现 `status` 列不再使用（假设逻辑迁移到了其他表），需要将其从 Schema 中移除。

```sql
-- 1. 删除列
ALTER TABLE dataspire_catalog.db_dev.orders_v1 DROP COLUMN status;

-- 2. 验证：查询表
-- 预期：status 列不再出现在结果中
SELECT * FROM dataspire_catalog.db_dev.orders_v1 LIMIT 1;

-- 3. 深入验证：检查底层文件
-- 即使 Schema 中删除了，旧的数据文件中依然物理存在 status 列的数据。
-- 只有运行 expire_snapshots 和 delete_orphan_files 后，这些无用数据才会被物理清除。
SELECT file_path, record_count 
FROM dataspire_catalog.db_dev.orders_v1.files;
```

---

### 高级案例：嵌套结构 (Struct) 的演进
Iceberg 对嵌套字段的支持同样强大。假设我们有一个 `address` 结构体。

```sql
-- 1. 添加包含 Struct 的列
ALTER TABLE dataspire_catalog.db_dev.orders_v1 ADD COLUMN address STRUCT<city: STRING, zip: STRING>;

-- 2. 写入带 Struct 的数据
INSERT INTO
    dataspire_catalog.db_dev.orders_v1
(order_id, user_id, amount, email, register_time, dt, address)
VALUES
    (1005, 505, 300.0, 'charlie@example.com', TIMESTAMP('2026-03-01'), '2026-03-13', NAMED_STRUCT('city', 'Beijing', 'zip', '100000'));

-- 3. 演进 Struct 内部：添加新字段 'street'
ALTER TABLE dataspire_catalog.db_dev.orders_v1
ADD COLUMNS (address.street STRING AFTER city);

-- 4. 验证嵌套演进
SELECT order_id, address.city, address.street 
FROM dataspire_catalog.db_dev.orders_v1 
WHERE order_id IN (1004, 1005);
-- 预期：1004 (旧数据) 的 street 为 NULL，1005 (新数据) 的 street 可能为 NULL (因为插入时没填) 或有值
```

---

## 最佳实践
### 始终使用列名而非位置
在 ETL 代码中，永远不要依赖 `SELECT *` 或列的位置索引（如 `row[0]`）。

+ 错误做法: `INSERT INTO target SELECT col1, col2 FROM source` (假设列顺序固定)
+ 正确做法: `INSERT INTO target (col1, col2, new_col) SELECT col1, col2, null FROM source`
+ 原因: 即使 Iceberg 能处理 Schema 演进，显式指定列名能防止因源表顺序变化导致的逻辑错误。

### 谨慎处理“不安全”的类型转换
Iceberg 只支持宽化转换 (Widening Conversion)，如 `INT` -> `BIGINT`。

+ 禁止: 直接 `ALTER COLUMN type STRING TO INT`。这会导致数据损坏或报错。
+ 解决方案: 如果需要缩小类型或进行复杂转换：
    1. 添加一个新列 `new_col` (目标类型)。
    2. 使用 `UPDATE` 或 `INSERT OVERWRITE` 将旧列 `CAST` 后写入新列。
    3. 验证数据无误后，删除旧列，重命名新列。

---

### 兼容性测试
在进行重大 Schema 变更（特别是涉及下游多个消费方）前：

1. 在开发环境 (`db_dev`) 演练。
2. 使用 `VERSION AS OF` 查询变更前的快照，确保旧作业仍能读取旧数据。
3. 确保下游引擎（如 Spark, Trino, Flink）的版本支持 Iceberg 的 Schema 演进特性。

### 文档化变更
虽然 Iceberg 记录了 `history`，但建议在表注释或外部文档中记录 Schema 变更的业务含义。

```sql
ALTER TABLE dataspire_catalog.db_dev.orders_v1
SET TBLPROPERTIES ('schema_change_log' = '2026-03-14: Renamed cust_id to user_id; Promoted amount to DOUBLE');
```

---

## 总结速查表
| 操作类型 | Spark SQL 语法模板 | 注意事项 |
| --- | --- | --- |
| **Add** | `ALTER TABLE <tbl> ADD COLUMNS (col_name TYPE [AFTER col]);` | 旧数据自动补 NULL |
| **Drop** | `ALTER TABLE <tbl> DROP COLUMN col_name;` | 物理数据需等待 GC 清理 |
| **Rename** | `ALTER TABLE <tbl> RENAME COLUMN old_name TO new_name;` | 列 ID 不变，最安全的操作 |
| **Update** | `ALTER TABLE <tbl> ALTER COLUMN col_name TYPE NEW_TYPE;` | 仅限安全转换 (如 Int->Long) |
| **Reorder** | `ALTER TABLE <tbl> ADD COLUMNS (struct_col.new_field TYPE);` | 支持多层嵌套 |


